# 05 — Associações estatísticas

Aplica Qui-quadrado e V de Cramér para medir a associação entre variáveis categóricas e a gravidade dos acidentes.

In [ ]:
import os
import numpy as np
import pandas as pd
from scipy.stats import chi2_contingency

PASTA_PROJETO = "TCC_PRF"
PASTA_PROCESSED = os.path.join(PASTA_PROJETO, "data", "processed")
PASTA_OUTPUT = os.path.join(PASTA_PROJETO, "outputs", "estatistica")
os.makedirs(PASTA_OUTPUT, exist_ok=True)

df = pd.read_parquet(os.path.join(PASTA_PROCESSED, "prf_acidentes_silver_2020_2025.parquet"))

In [ ]:
def cramers_v_corrigido(tabela):
    chi2 = chi2_contingency(tabela)[0]
    n = tabela.values.sum()
    if n <= 1:
        return np.nan
    phi2 = chi2/n
    r, k = tabela.shape
    phi2corr = max(0, phi2 - ((k-1)*(r-1))/(n-1))
    rcorr = r - ((r-1)**2)/(n-1)
    kcorr = k - ((k-1)**2)/(n-1)
    den = min(kcorr-1, rcorr-1)
    return np.nan if den <= 0 else np.sqrt(phi2corr/den)

def interpretar(v):
    if pd.isna(v): return "Não calculado"
    if v < .10: return "Muito fraca"
    if v < .20: return "Fraca"
    if v < .30: return "Moderada"
    if v < .50: return "Forte"
    return "Muito forte"

def analisar(dados, coluna, alvo="acidente_grave"):
    temp = dados[[coluna, alvo]].copy()
    temp[coluna] = temp[coluna].astype("string").fillna("Não informado")
    tab = pd.crosstab(temp[coluna], temp[alvo])
    if tab.shape[0] < 2:
        return None
    chi2, p, dof, expected = chi2_contingency(tab)
    v = cramers_v_corrigido(tab)
    return {
        "variavel": coluna,
        "n_registros": len(temp),
        "categorias": temp[coluna].nunique(),
        "chi2": chi2,
        "graus_liberdade": dof,
        "p_valor": p,
        "cramers_v": v,
        "interpretacao": interpretar(v),
        "pct_celulas_esperadas_lt5": (np.asarray(expected) < 5).mean()*100
    }

In [ ]:
variaveis_modelo_a = [
    "dia_semana","mes","hora","uf","br","fase_dia","sentido_via",
    "condicao_metereologica","tipo_pista","uso_solo","fim_semana"
]
variaveis_modelo_a += [c for c in df.columns if c.startswith("tracado_") and c != "tracado_via"]

resultados = []
for c in variaveis_modelo_a:
    r = analisar(df, c)
    if r:
        r["experimento"] = "Modelo A - contexto"
        resultados.append(r)

for c in ["tipo_acidente"]:
    r = analisar(df, c)
    r["experimento"] = "Modelo B - ocorrência"
    resultados.append(r)

df_causa = df[df["ano"] >= 2021].copy()
r = analisar(df_causa, "causa_acidente")
r["experimento"] = "Análise complementar - causa"
r["observacao"] = "Analisada somente entre 2021 e 2025 devido à alteração de taxonomia."
resultados.append(r)

resultados = pd.DataFrame(resultados).sort_values("cramers_v", ascending=False)
display(resultados[["variavel","experimento","categorias","p_valor","cramers_v","interpretacao","pct_celulas_esperadas_lt5"]])

resultados.to_csv(os.path.join(PASTA_OUTPUT, "associacoes_cramers_v.csv"),
                  index=False, encoding="utf-8-sig")